In [1]:
import math
import cv2
import numpy as np
import gymnasium as gym

from gymnasium import spaces

In [2]:
from importnb import Notebook
with Notebook():
    from LabCacheService import CacheService

cache = CacheService(capacity_bytes=2_000_000_000)

Video loaded successfully
Frame count: 1800
FPS: 30.0
Frame size: 3840 x 1920


KeyboardInterrupt: 

In [ ]:
from importnb import Notebook
with Notebook():
    from LabTrajectory import simulate_viewport_with_tiles, visualize_trajectory_sequence

if __name__ == "__main__":
    n_frames = 1_800
    num_tiles = 4

    yaw, pitch, tiles_per_frame, tiles_array = simulate_viewport_with_tiles(
        num_steps=n_frames,
        n=num_tiles,
        fov_yaw=120,
        fov_pitch=60,
        damping=0.99,
        step_size=1.5,
        start_yaw=180,
        start_pitch=0
    )
    user_tiles = tiles_per_frame[:60]
    # visualize_trajectory_sequence(
    #     yaw, 
    #     pitch, 
    #     user_tiles, 
    #     n=num_tiles, 
    #     fov_yaw=120, 
    #     fov_pitch=60,
    #     start_frame=0, 
    #     num_frames_to_show=60, 
    #     cols=6
    # )

In [ ]:
n = 4
def start_attention_tile_prob():
    # video_path = r'C:\Users\es25591\Workspace\360dataset\content\saliency\pacman_saliency.mp4'
    video_path = f"/home/eduardo/Videos/saliency/pacman_saliency.mp4"
    cap = cv2.VideoCapture(video_path)

    ret_f, f = cap.read()
    if not ret_f:
        return
    att = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)    
    
    def split_into_tiles(arr, n):
        h, w = arr.shape
        tile_h, tile_w = h // n, w // n
        tiles = []
        for i in range(n):
            for j in range(n):
                y0, y1 = i * tile_h, (i + 1) * tile_h
                x0, x1 = j * tile_w, (j + 1) * tile_w
                tiles.append(arr[y0:y1, x0:x1])
        return np.array(tiles)

    tiles = split_into_tiles(att, n)

    total_attention = np.sum(att)

    att_prob = [
        float(np.sum(tile) / total_attention) for tile in tiles
    ]
    
    return att_prob

In [ ]:
# CPT parameters (typical values from literature; adjustable)
alpha = 0.88  # gain curvature
beta = 0.88   # loss curvature
lam = 2.25    # loss aversion (losses weighted more)
prelec_gamma = 0.65  # probability weighting parameter (Prelec)
n = 4
num_tiles = n*n
class CPTPrefetchEnvSimple(gym.Env):
    def __init__(
        self, 
        num_tiles: int = num_tiles,
        user_tiles: np.ndarray = user_tiles,
        n_users: int = 1,
        cache_capacity: int = 10,
    ):
        super(CPTPrefetchEnvSimple, self).__init__()
        self.action_space = spaces.Discrete(2)  # Example: two discrete actions
        self.observation_space = spaces.Box(
            low=0, 
            high=1, 
            shape=(4,), 
            dtype=np.float32
        )  # Example: 4D continuous observation


        self.capacity = self.cache_capacity = cache_capacity
        self.sizes = np.random.randint(1, 4, size=num_tiles) 

        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -2.0

        self.gamma = 0.7
        self.alpha = 0.88
        self.beta = 0.88
        self.lam = 2.25

        self.step_count = 0
        self.user_tiles = user_tiles

        # self.current_p = np.zeros(num_tiles, dtype=float)
        self.current_p = start_attention_tile_prob()
        print(self.current_p)


    def update_current_p(self) -> np.ndarray:
        p = np.zeros(num_tiles, dtype=int)

        for tile in self.user_tiles[self.step_count]:
            tx, ty = tile[0] - 1, tile[1] - 1
            p[tx + ty*n] = 1

        return p

    def reset(self):
        self.state = np.random.rand(4)  # Example initial state
        return self.state

    def step(self, action):
        self.state = np.random.rand(4)
        reward     = 0.0
        done       = np.random.rand() > 0.95
        info       = {}

        n_prefetch = int(action.sum())
        info = {}

        # Enforce capacity constraint and compute penalty if exceeded
        selected = np.asarray(np.where(action == 1)[0], dtype=int)
        if selected.size > 0 and n_prefetch > self.capacity:
            to_drop = int(n_prefetch - self.capacity)
            priorities = np.asarray(self.current_p)[selected]  # priorities aligned with selected
            order = np.argsort(priorities, kind='stable')      # indices into selected, ascending priorities
            sel_sorted = selected[order]
            drop = sel_sorted[:to_drop]
            action[drop] = 0
            info["clipped_dropped_indices"] = drop.tolist()
            penalty = -0.5 * to_drop
            reward += penalty

        reward += self.compute_expected_cpt_reward(action)

        return self.state, reward, done, info

    def compute_expected_cpt_reward(self, action: np.ndarray) -> float:
        expected_reward = 0.0
        for tile in range(num_tiles):
            p = float(self.current_p[tile])
            w = self.prelec_w(p)
            if action[tile] == 1:
                expected_outcome = p * self.gain_if_prefetched
            else:
                expected_outcome = p * self.loss_if_not_prefetched
            
            v = self.cpt_value(expected_outcome, alpha=self.alpha, beta=self.beta, lam=self.lam)
            expected_reward += w * v

        return float(expected_reward)

    def u_gain(self, x):
        return x**alpha

    def u_loss(self, x):
        return -lam * ((-x)**beta)
    
    def prelec_w(self, p: float, alpha: float = 0.65) -> float:
        if p <= 0.0: return 0.0
        if p >= 1.0: return 1.0
        return math.exp(-((-math.log(p)) ** alpha))

    def cpt_value_function(
        self,
        x: float, 
        alpha: float = 0.88, 
        beta: float = None, 
        lam: float = 2.25, 
        x0: float = 0.0
    ) -> float:
        if beta is None:
            beta = alpha
        diff = x - x0
        if diff >= 0:
            return diff ** alpha
        else:
            return -lam * ((-diff) ** beta)

    def cpt_value(self, x: float, alpha: float = 0.88, beta: float = 0.88, lam: float = 2.25) -> float:
        if x >= 0:
            return x ** alpha
        else:
            return -lam * ((-x) ** beta)

In [ ]:
def map_user_tiles_to_action(step) -> np.ndarray:
    action = np.zeros(num_tiles, dtype=int)
    for tile in user_tiles[step]:
        tx, ty = tile[0] - 1, tile[1] - 1
        action[tx + ty*n] = 1
    return action

if __name__ == '__main__':
    env = CPTPrefetchEnvSimple(
        num_tiles=num_tiles,
        user_tiles=user_tiles,
        n_users=1,
        cache_capacity=50,
    )

    obs = env.reset()
    done = False
    total_reward = 0.0

    for step in range(60):
        action = map_user_tiles_to_action(step)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        print(f"Obs: {obs}, Reward: {reward}, Done: {done}")

    print(f"Total reward: {total_reward}")


[0.0008343041140910125, 0.02723348648278134, 0.02407229956387877, 0.0005747917525986161, 0.0006949913653000896, 0.2811378791377939, 0.1869168340207505, 0.0007377111462561423, 0.0008346325665210689, 0.23975113285164923, 0.19081337002375262, 0.000784693820453352, 0.0005634636379364598, 0.023069729968704843, 0.021688440744976792, 0.0002922388025552453]
Obs: [0.88456135 0.5144088  0.90578966 0.36554998], Reward: -0.7447469694904795, Done: False
Obs: [0.80137188 0.16089902 0.66550989 0.04831897], Reward: -0.7447469694904795, Done: False
Obs: [0.36233059 0.34848183 0.8344371  0.20140839], Reward: -0.7447469694904795, Done: False
Obs: [0.91797395 0.54705597 0.03897967 0.65214911], Reward: -0.7447469694904795, Done: False
Obs: [0.87122678 0.58122213 0.05242925 0.56854561], Reward: -0.7447469694904795, Done: False
Obs: [0.01557454 0.57225089 0.62318428 0.4869013 ], Reward: -0.7447469694904795, Done: False
Obs: [0.76759813 0.03740486 0.78271288 0.32632252], Reward: -0.7447469694904795, Done: Fal

In [ ]:
# Ready-to-run Python implementation of CPT-driven greedy knapsack for 360° tile prefetching.
# This will:
#  - generate synthetic candidate tiles (p_i, size, q_pref, q_nopref)
#  - implement Prelec probability weighting and CPT utility
#  - compute CPT subjective values V_i
#  - run greedy knapsack (by V_i/size) and exact DP knapsack (by value)
#  - compare performance: expected QoE, wasted bytes, prefetched bytes used
#
# To run locally: save as `cpt_prefetch.py` and run `python cpt_prefetch.py`.
# This cell will execute here and print a small report and table.
import math
import random
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple

random.seed(0)

# ---------------------------
# CPT / Prelec implementations
# ---------------------------

def prelec_weight(p: float, alpha: float = 0.65) -> float:
    """
    Prelec probability weighting function: w(p) = exp( -(-ln p)^alpha )
    alpha in (0,1] produces overweighting of small probabilities when alpha < 1.
    """
    if p <= 0.0:
        return 0.0
    if p >= 1.0:
        return 1.0
    return math.exp(-((-math.log(p)) ** alpha))

def cpt_value_function(x: float, alpha: float = 0.88, beta: float = None, lam: float = 2.25, x0: float = 0.0) -> float:
    """
    S-shaped CPT value function (Kahneman & Tversky style):
      u(x) = (x - x0)^alpha            if x >= x0
           = - lam * (x0 - x)^beta     if x < x0
    Defaults set beta = alpha if None.
    """
    if beta is None:
        beta = alpha
    diff = x - x0
    if diff >= 0:
        return diff ** alpha
    else:
        return -lam * ((-diff) ** beta)

# ---------------------------
# Problem setup: tiles & QoE
# ---------------------------

@dataclass
class Tile:
    id: int
    p: float            # predicted probability of being viewed
    size_kb: int        # size to prefetch at target resolution (KB)
    q_pref: float       # QoE if prefetched and viewed (e.g., VMAF)
    q_nopref: float     # QoE if not prefetched and viewed (fallback)
    
def generate_synthetic_tiles(n_tiles: int = 40) -> List[Tile]:
    tiles = []
    for i in range(n_tiles):
        # probabilities skewed: most tiles low prob, few high prob (typical 360°)
        p = min(0.99, max(0.01, random.betavariate(0.8, 3.0)))
        # size between 50 KB and 800 KB
        size_kb = random.randint(50, 800)
        # q_pref: higher quality metric if prefetched (simulate VMAF-like)
        q_pref = random.uniform(55, 95)  # high-quality delivered
        # q_nopref: fallback quality (lower)
        q_nopref = max(10.0, q_pref - random.uniform(6, 30))
        tiles.append(Tile(i, p, size_kb, q_pref, q_nopref))
    return tiles

# ---------------------------
# Value computations & knapsack
# ---------------------------

def compute_tile_cpt_value(tile: Tile, prelec_alpha: float, value_alpha: float, value_beta: float, lam: float, x0: float) -> float:
    """
    Compute marginal CPT subjective value of prefetching this tile.
    Using simplified two-outcome approximation:
      V_i = w(p_i) * [ u(q_pref - x0) - u(q_nopref - x0) ]
    """
    w = prelec_weight(tile.p, alpha=prelec_alpha)
    u_pref = cpt_value_function(tile.q_pref, alpha=value_alpha, beta=value_beta, lam=lam, x0=x0)
    u_nopref = cpt_value_function(tile.q_nopref, alpha=value_alpha, beta=value_beta, lam=lam, x0=x0)
    return w * (u_pref - u_nopref)

def greedy_cpt_knapsack(tiles: List[Tile], cache_kb: int, **cpt_params) -> Tuple[List[Tile], int]:
    """
    Greedy selection by CPT value per KB (V_i / size_kb).
    Returns selected tiles list and total prefetched KB.
    """
    scored = []
    for t in tiles:
        v = compute_tile_cpt_value(t, **cpt_params)
        scored.append((t, v))
    # Filter out non-positive subjective values
    scored = [(t, v) for t, v in scored if v > 0]
    scored.sort(key=lambda tv: (tv[1] / tv[0].size_kb), reverse=True)
    
    selected = []
    used = 0
    for t, v in scored:
        if used + t.size_kb <= cache_kb:
            selected.append(t)
            used += t.size_kb
    return selected, used

def dp_knapsack_exact(tiles: List[Tile], cache_kb: int, **cpt_params) -> Tuple[List[Tile], int]:
    """
    Exact 0-1 knapsack via dynamic programming. Values are CPT subjective values.
    Note: sizes and capacities are integers (KB). Complexity O(n * cache_kb).
    For moderate cache_kb this is fine; otherwise use greedy.
    """
    n = len(tiles)
    # compute values
    values = [max(0.0, compute_tile_cpt_value(t, **cpt_params)) for t in tiles]
    weights = [t.size_kb for t in tiles]
    W = cache_kb
    # DP table: store max value achievable with first i items and capacity w
    # To save memory, use 1D DP with reconstruction via keep table
    dp = [0.0] * (W + 1)
    keep = [[False]*(W + 1) for _ in range(n)]
    for i in range(n):
        wi = weights[i]
        vi = values[i]
        if wi > W or vi <= 0:
            continue
        # traverse backwards
        for w in range(W, wi-1, -1):
            if dp[w - wi] + vi > dp[w]:
                dp[w] = dp[w - wi] + vi
                keep[i][w] = True
    # reconstruct choices
    w = W
    chosen = []
    for i in range(n-1, -1, -1):
        if keep[i][w]:
            chosen.append(tiles[i])
            w -= weights[i]
    chosen.reverse()
    used = sum(t.size_kb for t in chosen)
    return chosen, used

# ---------------------------
# Evaluation metrics
# ---------------------------

def evaluate_selection(tiles: List[Tile], selected: List[Tile]) -> dict:
    """
    Compute expected QoE (expected viewed QoE across tiles), bytes wasted,
    and prefetched-used bytes ratio. Assumes viewing events independent.
    Expected QoE = sum_i [ p_i * (q_pref if prefetched else q_nopref) + (1-p_i)*(some baseline) ]
    For simplicity baseline for unviewed branch not counted toward QoE.
    We'll compute:
      - expected_viewport_QoE = sum_i p_i * q_view_i
      - expected_wasted_kb = sum_{i in selected} (1 - p_i) * size_kb
    """
    selected_ids = {t.id for t in selected}
    expected_qoe = sum(t.p * (t.q_pref if t.id in selected_ids else t.q_nopref) for t in tiles)
    prefetched_kb = sum(t.size_kb for t in selected)
    wasted_kb = sum((1 - t.p) * t.size_kb for t in selected)
    used_kb_expected = sum(t.p * t.size_kb for t in selected)
    return {
        "expected_qoe": expected_qoe,
        "prefetched_kb": prefetched_kb,
        "wasted_kb_expected": wasted_kb,
        "expected_used_kb": used_kb_expected
    }

# ---------------------------
# Main demo run
# ---------------------------

def demo_run(n_tiles:int=40, cache_kb:int=5000):
    tiles = generate_synthetic_tiles(n_tiles)
    print(f"Generated {tiles} synthetic tiles.")

    # CPT params (reasonable defaults; you can tune these)
    cpt_params = {
        "prelec_alpha": 0.65,   # <1 overweight small probabilities
        "value_alpha": 0.88,    # concave gains
        "value_beta": 0.88,     # convex losses
        "lam": 2.25,            # loss aversion
        "x0": 50.0              # reference QoE (e.g., baseline VMAF)
    }
    
    # compute greedy selection
    greedy_sel, greedy_used = greedy_cpt_knapsack(tiles, cache_kb, **cpt_params)
    greedy_eval = evaluate_selection(tiles, greedy_sel)
    
    # compute exact DP selection (may be slower)
    dp_sel, dp_used = dp_knapsack_exact(tiles, cache_kb, **cpt_params)
    dp_eval = evaluate_selection(tiles, dp_sel)
    
    # compute a probability-only knapsack baseline: value = p * delta_q (no CPT)
    def prob_value(t: Tile) -> float:
        return t.p * (t.q_pref - t.q_nopref)
    prob_scored = [(t, prob_value(t)) for t in tiles if prob_value(t) > 0]
    prob_scored.sort(key=lambda tv: (tv[1] / tv[0].size_kb), reverse=True)
    prob_selected = []
    used = 0
    for t, v in prob_scored:
        if used + t.size_kb <= cache_kb:
            prob_selected.append(t)
            used += t.size_kb
    prob_eval = evaluate_selection(tiles, prob_selected)
    
    # Prepare DataFrame summarizing top tiles and selections
    df = pd.DataFrame([{
        "id": t.id,
        "p": t.p,
        "size_kb": t.size_kb,
        "q_pref": round(t.q_pref,2),
        "q_nopref": round(t.q_nopref,2),
        "delta_q": round(t.q_pref - t.q_nopref,2),
        "cpt_value": round(compute_tile_cpt_value(t, **cpt_params),4),
        "prob_value": round(prob_value(t),4),
        "selected_greedy": (t in greedy_sel),
        "selected_dp": (t in dp_sel),
        "selected_prob": (t in prob_selected)
    } for t in tiles])
    df = df.sort_values(by="cpt_value", ascending=False).reset_index(drop=True)
    
    # Print summary
    print("CPT params:", cpt_params)
    print(f"Cache budget: {cache_kb} KB")
    print("\n--- Greedy CPT knapsack ---")
    print(f"Prefetched KB: {greedy_eval['prefetched_kb']}, Expected QoE: {greedy_eval['expected_qoe']:.2f}, Expected wasted KB: {greedy_eval['wasted_kb_expected']:.2f}")
    print("\n--- Exact DP knapsack ---")
    print(f"Prefetched KB: {dp_eval['prefetched_kb']}, Expected QoE: {dp_eval['expected_qoe']:.2f}, Expected wasted KB: {dp_eval['wasted_kb_expected']:.2f}")
    print("\n--- Probability-only knapsack baseline ---")
    print(f"Prefetched KB: {prob_eval['prefetched_kb']}, Expected QoE: {prob_eval['expected_qoe']:.2f}, Expected wasted KB: {prob_eval['wasted_kb_expected']:.2f}")
    
    # show top 12 tiles table
    display_df = df.head(12).copy()
    display_df.index += 1
    # Use caas_jupyter_tools if available, otherwise fall back to IPython.display
    try:
        import caas_jupyter_tools as jt
        jt.display_dataframe_to_user("Top_tiles_cpt_vs_prob", display_df)
    except Exception:
        from IPython.display import display
        print("Note: 'caas_jupyter_tools' not available — showing dataframe with IPython.display")
        display(display_df)
    
    return {
        "tiles_df": df,
        "greedy_eval": greedy_eval,
        "dp_eval": dp_eval,
        "prob_eval": prob_eval,
        "greedy_selected": greedy_sel,
        "dp_selected": dp_sel,
        "prob_selected": prob_selected,
        "cpt_params": cpt_params
        }

# Run demo
results = demo_run(n_tiles=40, cache_kb=5000)


Generated [Tile(id=0, p=0.34669507555988033, size_kb=573, q_pref=74.4371078625125, q_nopref=46.39948389966934), Tile(id=1, p=0.13413081423954148, size_kb=306, q_pref=94.49036804132052, q_nopref=75.70884082319587), Tile(id=2, p=0.11492212553388491, size_kb=388, q_pref=73.88570861810854, q_nopref=65.46887962446776), Tile(id=3, p=0.05363510964570119, size_kb=538, q_pref=72.70773436585364, q_nopref=54.19524714619354), Tile(id=4, p=0.01, size_kb=734, q_pref=80.01063317094528, q_nopref=59.3251031354058), Tile(id=5, p=0.21457355882647974, size_kb=114, q_pref=62.64268366009562, q_nopref=43.022425885199496), Tile(id=6, p=0.5314012044163337, size_kb=570, q_pref=94.91886524344498, q_nopref=77.17598249536726), Tile(id=7, p=0.19716842852420477, size_kb=603, q_pref=63.12807409877733, q_nopref=37.94182342348312), Tile(id=8, p=0.37662346361008636, size_kb=374, q_pref=78.02604056659554, q_nopref=65.05813250892935), Tile(id=9, p=0.04473061602156032, size_kb=537, q_pref=57.76340993935595, q_nopref=35.474